# ResearchLanka Kaggle Main-Branch Full Run

Import this notebook into Kaggle and run cells from top to bottom.

It will:

- clone/pull the latest `main` branch
- copy your uploaded raw dataset into the repo
- install Python + Dagster dependencies
- run the Dagster no-collection preprocessing job
- build best-quality embeddings
- train Logistic Regression
- train Linear SVM
- compare model metrics
- zip outputs for download

It does **not** collect data from APIs or repositories.

## 1. Settings

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data")
OUTPUT_ZIP = WORK_DIR / "researchlanka-kaggle-outputs.zip"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)
print("Output zip:", OUTPUT_ZIP)

Repo: https://github.com/krish-anu/researchlanka-ai.git
Branch: main
Code dir: /kaggle/working/code
Backend dir: /kaggle/working/code/backend
Dataset data dir: /kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data
Output zip: /kaggle/working/researchlanka-kaggle-outputs.zip


## 2. Check Kaggle Dataset Exists

If this fails, your Kaggle dataset path is different. Update `DATASET_DATA_DIR` above.

In [2]:
!ls -la /kaggle/input
!find /kaggle/input -maxdepth 5 -type d | head -80
!test -d {DATASET_DATA_DIR} && echo "Dataset path OK" || echo "Dataset path NOT FOUND"

total 12
drwxr-xr-x 3 root root 4096 Aug 20 12:39 .
drwxr-xr-x 8 root root 4096 Aug 20 12:39 ..
drwxr-xr-x 3 root root 4096 Aug 20 12:39 datasets
/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/anusankrishnathas
/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data
/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend
/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data
Dataset path OK


## 3. Clone Or Pull Latest Main Branch

In [3]:
%cd /kaggle/working
if CODE_DIR.exists() and (CODE_DIR / ".git").exists():
    %cd /kaggle/working/code
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} code
    %cd /kaggle/working/code

!git log --oneline -3
!ls

/kaggle/working
Cloning into 'code'...
remote: Enumerating objects: 2764, done.
remote: Counting objects: 100% (559/559), done.
remote: Compressing objects: 100% (365/365), done.
remote: Total 2764 (delta 244), reused 348 (delta 176), pack-reused 2205 (from 1)
Receiving objects: 100% (2764/2764), 11.85 MiB | 11.33 MiB/s, done.
Resolving deltas: 100% (1642/1642), done.
/kaggle/working/code
4247624 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #461 from krish-anu/feature/machine-learning
3fb34c3 Fix: Kaggle notebook package install and CSV inspection cell
a842fdf Merge pull request #460 from krish-anu/feature/machine-learning
backend  CHANGELOG.md  frontend  Makefile  notebooks  README.md


## 4. Copy Uploaded Raw Data Into Backend

In [4]:
%cd /kaggle/working/code/backend
!rm -rf data
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -60

/kaggle/working/code/backend
data/raw/uwu/rest_items.jsonl
data/raw/jfn_medicine/html_meta.jsonl
data/raw/pdn/rest_items.jsonl
data/raw/vpa/oai_dc.jsonl
data/raw/ruh/oai_dc.jsonl
data/raw/seu/oai_dc.jsonl
data/raw/sljol/crossref_works.jsonl
data/raw/uom/oai_dc.jsonl
data/raw/nsf/rest_items.jsonl
data/raw/rjt/oai_dc.jsonl
data/raw/esn/oai_dc.jsonl
data/raw/sltc/oai_dc.jsonl
data/raw/ou/oai_dc.jsonl
data/raw/vau/oai_dc.jsonl
data/raw/cmb/rest_items.jsonl
data/raw/sliit/oai_dc.jsonl
data/raw/jfn_research/html_meta.jsonl
data/raw/openalex/openalex_sri_lanka_doi_conflicts.csv
data/raw/openalex/openalex_sri_lanka_works.csv
data/raw/openalex/openalex_sri_lanka_works.jsonl
data/raw/openalex/openalex_sri_lanka_works_progress.json
data/raw/openalex/openalex_sri_lanka_pagination_audit.json
data/raw/busl/rest_items.jsonl
data/processed/crossref/crossref_sri_lanka_works.csv
data/processed/crossref/crossref_sri_lanka_works.jsonl


## 5. Install Dependencies

Kaggle may show dependency conflict warnings. Continue if the install completes. If the session restarts, rerun from the top.

In [5]:
%cd /kaggle/working/code/backend
!pip install $(grep -v '^psycopg2==' requirements.txt)
!pip install dagster==1.13.16 dagster-webserver
!pip install -e dagster-quickstart
!python -m dagster --version

/kaggle/working/code/backend
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 787.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 578.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 6. Run Dagster Pipeline Without Data Collection

This prepares existing source files and runs preprocessing through the analysis-ready dataset.

In [6]:
%cd /kaggle/working/code/backend/dagster-quickstart
!python -m dagster job execute \
  -m dagster_quickstart.definitions \
  -j researchlanka_no_collection_preprocessing_job

/kaggle/working/code/backend/dagster-quickstart
/usr/local/lib/python3.12/dist-packages/click/core.py:853: SupersessionWarning: Function `job_execute_command` is superseded and its usage is discouraged. Use 'dg launch --job <job_name>' instead.
  return callback(*args, **kwargs)

  Telemetry:

  As an open-source project, we collect usage statistics to inform development priorities. For more
  information, read https://docs.dagster.io/about/telemetry.

  We will not see or store any data that is processed by your code.

  To opt-out, add the following to $DAGSTER_HOME/dagster.yaml, creating that file if necessary:

    telemetry:
      enabled: false


  Welcome to Dagster!

  If you have any questions or would like to engage with the Dagster team, please join us on Slack
  (https://bit.ly/39dvSsF).

2026-08-20 12:42:00 +0000 - dagster - DEBUG - researchlanka_no_collection_preprocessing_job - 9009a2b6-20db-4b73-bfa1-e346985b270e - 126 - RUN_START - Started execution of run for "researc

## 7. Verify Preprocessing Outputs

In [7]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv
!ls -lh data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

import pandas as pd
paths = [
    'data/processed/common/common_publications_final.csv',
    'data/processed/common/common_publications_final_2016_2026_analysis_ready.csv',
]
for path in paths:
    frame = pd.read_csv(path, nrows=5)
    total = sum(1 for _ in open(path, encoding='utf-8')) - 1
    print(path, 'rows=', total, 'columns=', len(frame.columns))


/kaggle/working/code/backend
-rw-r--r-- 1 root root 105M Aug 20 12:42 data/processed/repositories_combined.csv
-rw-r--r-- 1 root root 48M Aug 20 12:42 data/processed/sljol.csv
-rw-r--r-- 1 root root 292M Aug 20 13:01 data/processed/common/common_publications_final.csv
-rw-r--r-- 1 root root 388M Aug 20 13:09 data/processed/common/common_publications_final_2016_2026_analysis_ready.csv
data/processed/common/common_publications_final.csv rows= 182149 columns= 56
data/processed/common/common_publications_final_2016_2026_analysis_ready.csv rows= 147859 columns= 71


## 8. Build Best-Quality Embeddings

Full text fields, trigrams, larger vocabulary, 512 dimensions, no row limit.

In [8]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python \
  EMBED_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  EMBED_MAX_FEATURES=100000 \
  EMBED_NGRAM_MAX=3 \
  EMBED_DIM=512
!ls -lh data/models/publication_text_embeddings.parquet data/models/publication_text_embedding_model.joblib data/models/publication_text_embeddings_summary.txt

/kaggle/working/code/backend
python scripts/modeling/generate_publication_text_embeddings.py --input data/processed/common/common_publications_final.csv --output data/models/publication_text_embeddings.parquet --model-output data/models/publication_text_embedding_model.joblib --manifest-output data/models/publication_text_embeddings_manifest.json --summary-output data/models/publication_text_embeddings_summary.txt --text-columns title,abstract,topics,keywords,concepts --metadata-columns record_number,publication_year,title,doi,openalex_id,source_dataset,source_institution_id,source_record_id --embedding-dim 512 --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 
Generated publication text embeddings: rows=182139, dim=512, output=data/models/publication_text_embeddings.parquet
-rw------- 1 root root 395M Aug 20 13:17 data/models/publication_text_embedding_model.joblib
-rw-r--r-- 1 root root 560M Aug 20 13:17 data/models/publication_text_embeddings.parquet
-rw------- 1 root roo

## 9. Train Best-Quality Logistic Regression

In [9]:
%cd /kaggle/working/code/backend
!make train-logreg PYTHON=python \
  LOGREG_TEXT_COLUMNS=title,abstract,topics,keywords,concepts \
  LOGREG_MAX_FEATURES=100000 \
  LOGREG_NGRAM_MAX=3 \
  LOGREG_MAX_ITER=2000
!cat data/models/logistic_regression_primary_domain_metrics.txt

/kaggle/working/code/backend
python scripts/modeling/train_logistic_regression_classifier.py --input data/processed/common/common_publications_final.csv --label-column primary_domain --text-columns title,abstract,topics,keywords,concepts --model-output data/models/logistic_regression_primary_domain.joblib --metrics-output data/models/logistic_regression_primary_domain_metrics.txt --label-counts-output data/models/logistic_regression_primary_domain_labels.csv --predictions-output data/models/logistic_regression_primary_domain_predictions.csv --manifest-output data/models/logistic_regression_primary_domain_manifest.json --max-features 100000 --min-df 2 --max-df 0.95 --ngram-max 3 --min-class-count 20 --test-size 0.2 --max-iter 2000 
Trained Logistic Regression classifier on 72,624 rows.
Classes: 4
Accuracy: 0.8687
Macro F1: 0.8620
Model: data/models/logistic_regression_primary_domain.joblib
Model SHA-256: c52435bc3460078e20b212943d13c80e607f3dba6422c0fd2dcdd8b3d5f1ec6a
Metrics: data/mode

In [10]:
%cd /kaggle/working/code/backend
!pip install -e . --no-deps

/kaggle/working/code/backend
Obtaining file:///kaggle/working/code/backend
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for research-analytics-framework (pyproject.toml) ... done
  Created wheel for research-analytics-framework: filename=research_analytics_framework-0.1.0-0.editable-py3-none-any.whl size=7978 sha256=9defcf58f45f21fc2c075570192c64bf96608f7689253f2be19c9498ad0e6f97
  Stored in directory: /tmp/pip-ephem-wheel-cache-yf5a18_w/wheels/48/6f/98/3205cf08ddaa25af658fb7d96745a6be29b5c675f81653f651
Successfully built research-analytics-framework


## 10. Train Best-Quality Linear SVM

If Kaggle RAM fails, rerun this cell with `--max-features 50000`.

In [11]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,topics,keywords,concepts \
  --ngram-max 3 \
  --max-features 100000 \
  --c-values 0.1,1,10 \
  --cv-folds 3 \
  --class-weight balanced \
  --max-iter 5000
!cat data/models/linear_svm_primary_domain_metrics.txt

/kaggle/working/code/backend

LINEAR SVM TRAINING COMPLETED
Best C        : 1.0
CV macro F1   : 0.8773
Accuracy      : 0.8880
Macro F1      : 0.8811
Weighted F1   : 0.8881

Artifacts:
Model         : /kaggle/working/code/backend/data/models/linear_svm_primary_domain.joblib
Metrics       : /kaggle/working/code/backend/data/models/linear_svm_primary_domain_metrics.txt
Predictions   : /kaggle/working/code/backend/data/models/linear_svm_primary_domain_predictions.csv
Manifest      : /kaggle/working/code/backend/data/models/linear_svm_primary_domain_manifest.json
Publication Linear SVM Classifier

model_family: linear_svm
input_csv: data/processed/common/common_publications_final.csv
label_column: primary_domain
text_columns: title, abstract, topics, keywords, concepts

ngram_max: 3
best_C: 1.0

input_rows: 182149
usable_rows: 72624
train_rows: 61730
test_rows: 10894

class_count: 4

cv_macro_f1: 0.8773
accuracy: 0.8880
macro_f1: 0.8811
weighted_f1: 0.8881

Class Distribution:
Physical Scie

## 11. Compare Models

In [12]:
%cd /kaggle/working/code/backend
!grep -E "model_family|label_column|accuracy|macro_f1|weighted_f1" data/models/*metrics.txt || true

/kaggle/working/code/backend
data/models/linear_svm_primary_domain_metrics.txt:model_family: linear_svm
data/models/linear_svm_primary_domain_metrics.txt:label_column: primary_domain
data/models/linear_svm_primary_domain_metrics.txt:cv_macro_f1: 0.8773
data/models/linear_svm_primary_domain_metrics.txt:accuracy: 0.8880
data/models/linear_svm_primary_domain_metrics.txt:macro_f1: 0.8811
data/models/linear_svm_primary_domain_metrics.txt:weighted_f1: 0.8881
data/models/linear_svm_primary_domain_metrics.txt:         accuracy                           0.89     10894
data/models/logistic_regression_primary_domain_metrics.txt:model_family: logistic_regression
data/models/logistic_regression_primary_domain_metrics.txt:label_column: primary_domain
data/models/logistic_regression_primary_domain_metrics.txt:accuracy: 0.8687
data/models/logistic_regression_primary_domain_metrics.txt:macro_f1: 0.8620
data/models/logistic_regression_primary_domain_metrics.txt:weighted_f1: 0.8691
data/models/logistic_r

## 12. Zip Outputs For Download

In [13]:
%cd /kaggle/working/code/backend
!rm -f /kaggle/working/researchlanka-kaggle-outputs.zip
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip

/kaggle/working/code/backend
  adding: data/processed/ (stored 0%)
  adding: data/processed/repositories/ (stored 0%)
  adding: data/processed/repositories/jfn_research.jsonl (deflated 78%)
  adding: data/processed/repositories/uom.jsonl (deflated 72%)
  adding: data/processed/repositories/seu.jsonl (deflated 75%)
  adding: data/processed/repositories/nsf.jsonl (deflated 81%)
  adding: data/processed/repositories/busl.jsonl (deflated 85%)
  adding: data/processed/repositories/jfn_medicine.jsonl (deflated 89%)
  adding: data/processed/repositories/cmb.jsonl (deflated 71%)
  adding: data/processed/repositories/sliit.jsonl (deflated 71%)
  adding: data/processed/repositories/ruh.jsonl (deflated 73%)
  adding: data/processed/repositories_combined.csv (deflated 69%)
  adding: data/processed/sljol.csv (deflated 79%)
  adding: data/processed/crossref/ (stored 0%)
  adding: data/processed/crossref/crossref_sri_lanka_works.csv (deflated 78%)
  adding: data/processed/crossref/crossref_sri_lanka_

Download this file from the Kaggle output panel:

```text
/kaggle/working/researchlanka-kaggle-outputs.zip
```

In [14]:
import os

for name in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", name)
    size_gb = os.path.getsize(path) / (1024**3) if os.path.isfile(path) else 0
    kind = "file" if os.path.isfile(path) else "folder"
    print(kind, name, round(size_gb, 3), "GB")

file researchlanka-kaggle-outputs.zip 2.128 GB
file __notebook__.ipynb 0.0 GB
folder code 0 GB


In [15]:
import os

source_file = "/kaggle/working/researchlanka-kaggle-outputs.zip"
output_dir = "/kaggle/working/split_outputs"
os.makedirs(output_dir, exist_ok=True)

chunk_size = 500 * 1024 * 1024

with open(source_file, "rb") as f:
    part = 1
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break

        part_path = os.path.join(output_dir, f"researchlanka_outputs_part_{part:03d}.zip.part")
        with open(part_path, "wb") as out:
            out.write(chunk)

        print("Created:", part_path, round(os.path.getsize(part_path) / (1024**2), 1), "MB")
        part += 1

Created: /kaggle/working/split_outputs/researchlanka_outputs_part_001.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_002.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_003.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_004.zip.part 500.0 MB
Created: /kaggle/working/split_outputs/researchlanka_outputs_part_005.zip.part 179.0 MB


In [16]:
import os
import zipfile
import glob

# Find model folders
for path in glob.glob("/kaggle/working/**/data/models", recursive=True):
    print("FOUND:", path)

FOUND: /kaggle/working/code/backend/data/models


In [17]:
import zipfile
import os

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    names = src.namelist()
    
    model_files = [
        name for name in names
        if "/data/models/" in name or name.startswith("data/models/")
    ]
    
    print("Model files found:", len(model_files))
    for name in model_files[:20]:
        print(name)

    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))

Model files found: 15
data/models/
data/models/publication_text_embeddings.parquet
data/models/logistic_regression_primary_domain_labels.csv
data/models/linear_svm_primary_domain_metrics.txt
data/models/logistic_regression_primary_domain.joblib
data/models/linear_svm_primary_domain.joblib
data/models/publication_text_embedding_model.joblib
data/models/publication_text_embeddings_manifest.json
data/models/linear_svm_primary_domain_labels.csv
data/models/logistic_regression_primary_domain_manifest.json
data/models/linear_svm_primary_domain_manifest.json
data/models/publication_text_embeddings_summary.txt
data/models/logistic_regression_primary_domain_metrics.txt
data/models/logistic_regression_primary_domain_predictions.csv
data/models/linear_svm_primary_domain_predictions.csv
Created: /kaggle/working/researchlanka-models-only.zip
Size MB: 905.27


In [18]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-models-only.zip"))

/kaggle/working/researchlanka-models-only.zip

In [19]:
import zipfile

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"

with zipfile.ZipFile(source_zip, "r") as z:
    for name in z.namelist()[:100]:
        print(name)

data/processed/
data/processed/repositories/
data/processed/repositories/jfn_research.jsonl
data/processed/repositories/uom.jsonl
data/processed/repositories/seu.jsonl
data/processed/repositories/nsf.jsonl
data/processed/repositories/busl.jsonl
data/processed/repositories/jfn_medicine.jsonl
data/processed/repositories/cmb.jsonl
data/processed/repositories/sliit.jsonl
data/processed/repositories/ruh.jsonl
data/processed/repositories_combined.csv
data/processed/sljol.csv
data/processed/crossref/
data/processed/crossref/crossref_sri_lanka_works.csv
data/processed/crossref/crossref_sri_lanka_works.jsonl
data/processed/common/
data/processed/common/publication_count_audit.csv
data/processed/common/common_publications_final_2016_2026.csv
data/processed/common/publication_references.csv
data/processed/common/common_publications_final_2016_2026_analysis_ready.csv
data/processed/common/publication_multivalue_items_2016_2026.csv
data/processed/common/common_publications_deduplicated.csv
data/pro

In [20]:
import zipfile
import os

source_zip = "/kaggle/working/researchlanka-kaggle-outputs.zip"
models_zip = "/kaggle/working/researchlanka-models-only.zip"

with zipfile.ZipFile(source_zip, "r") as src:
    model_files = [
        name for name in src.namelist()
        if name.startswith("data/models/") and not name.endswith("/")
    ]

    print("Model files found:", len(model_files))

    with zipfile.ZipFile(models_zip, "w", zipfile.ZIP_DEFLATED) as dst:
        for name in model_files:
            dst.writestr(name, src.read(name))
            print("Added:", name)

print("Created:", models_zip)
print("Size MB:", round(os.path.getsize(models_zip) / (1024**2), 2))

Model files found: 14
Added: data/models/publication_text_embeddings.parquet
Added: data/models/logistic_regression_primary_domain_labels.csv
Added: data/models/linear_svm_primary_domain_metrics.txt
Added: data/models/logistic_regression_primary_domain.joblib
Added: data/models/linear_svm_primary_domain.joblib
Added: data/models/publication_text_embedding_model.joblib
Added: data/models/publication_text_embeddings_manifest.json
Added: data/models/linear_svm_primary_domain_labels.csv
Added: data/models/logistic_regression_primary_domain_manifest.json
Added: data/models/linear_svm_primary_domain_manifest.json
Added: data/models/publication_text_embeddings_summary.txt
Added: data/models/logistic_regression_primary_domain_metrics.txt
Added: data/models/logistic_regression_primary_domain_predictions.csv
Added: data/models/linear_svm_primary_domain_predictions.csv
Created: /kaggle/working/researchlanka-models-only.zip
Size MB: 905.27


In [21]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/researchlanka-models-only.zip"))

/kaggle/working/researchlanka-models-only.zip